# Notebook 04: Raw-to-Bronze Ingestion

## Objective

This notebook creates the Bronze representation of the PaySim transaction
dataset.

The Bronze layer preserves all source columns and adds operational metadata
required for lineage, auditing, reconciliation, and idempotent processing.

This notebook will:

- ingest the raw CSV using an explicit Spark schema;
- validate the source structure;
- generate a unique pipeline run identifier;
- add ingestion timestamp and source-file metadata;
- generate a deterministic record hash;
- reconcile source and Bronze record counts;
- generate pipeline audit metrics;
- identify potential duplicate hashes.

Physical Parquet persistence is deferred because the current native Windows
Spark environment does not include Hadoop Windows filesystem binaries.

In [1]:
import os
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

In [2]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [3]:
spark = (
    SparkSession.builder
    .appName("PaySimRawToBronze")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Python executable:", sys.executable)

Spark version: 4.2.0
Spark master: local[4]
Python executable: c:\Projects\paysim-financial-data-pipeline\.venv\Scripts\python.exe


In [5]:
PROJECT_ROOT = Path(
    r"C:\Projects\paysim-financial-data-pipeline"
)

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "PS_20174392719_1491204439457_log.csv"
)

AUDIT_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "gold"
    / "pipeline_audit"
)

AUDIT_OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

print("Raw source:", RAW_DATA_PATH)
print("Source exists:", RAW_DATA_PATH.exists())

Raw source: C:\Projects\paysim-financial-data-pipeline\data\raw\PS_20174392719_1491204439457_log.csv
Source exists: True


In [6]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Source file was not found: {RAW_DATA_PATH}"
    )

In [7]:
transaction_schema = StructType(
    [
        StructField("step", IntegerType(), nullable=True),
        StructField("type", StringType(), nullable=True),
        StructField("amount", DoubleType(), nullable=True),
        StructField("nameOrig", StringType(), nullable=True),
        StructField("oldbalanceOrg", DoubleType(), nullable=True),
        StructField("newbalanceOrig", DoubleType(), nullable=True),
        StructField("nameDest", StringType(), nullable=True),
        StructField("oldbalanceDest", DoubleType(), nullable=True),
        StructField("newbalanceDest", DoubleType(), nullable=True),
        StructField("isFraud", IntegerType(), nullable=True),
        StructField("isFlaggedFraud", IntegerType(), nullable=True),
    ]
)

In [8]:
pipeline_run_id = str(uuid.uuid4())

pipeline_start_timestamp = datetime.now(
    timezone.utc
)

ingestion_date = pipeline_start_timestamp.date().isoformat()

source_filename = RAW_DATA_PATH.name

print("Pipeline run ID:", pipeline_run_id)
print("Pipeline start:", pipeline_start_timestamp)
print("Ingestion date:", ingestion_date)
print("Source filename:", source_filename)

Pipeline run ID: 8fb9eff3-1189-40a8-9463-883dbe445a45
Pipeline start: 2026-07-24 18:20:48.618880+00:00
Ingestion date: 2026-07-24
Source filename: PS_20174392719_1491204439457_log.csv


In [9]:
raw_df = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .schema(transaction_schema)
    .csv(str(RAW_DATA_PATH))
)

In [11]:
raw_df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



In [12]:
raw_df.show(
    n=5,
    truncate=False,
)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|type    |amount  |nameOrig   |oldbalanceOrg|newbalanceOrig|nameDest   |oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|1   |PAYMENT |9839.64 |C1231006815|170136.0     |160296.36     |M1979787155|0.0           |0.0           |0      |0             |
|1   |PAYMENT |1864.28 |C1666544295|21249.0      |19384.72      |M2044282225|0.0           |0.0           |0      |0             |
|1   |TRANSFER|181.0   |C1305486145|181.0        |0.0           |C553264065 |0.0           |0.0           |1      |0             |
|1   |CASH_OUT|181.0   |C840083671 |181.0        |0.0           |C38997010  |21182.0       |0.0           |1      |0             |
|1   |PAYMENT |11668.14|C2048537720|41554.0      |29885.86      |M1230701703|0.0   

In [13]:
source_record_count = raw_df.count()

print(
    "Source record count:",
    f"{source_record_count:,}",
)

Source record count: 6,362,620


In [14]:
EXPECTED_COLUMNS = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud",
    "isFlaggedFraud",
]

In [15]:
actual_columns = raw_df.columns

missing_columns = sorted(
    set(EXPECTED_COLUMNS) - set(actual_columns)
)

unexpected_columns = sorted(
    set(actual_columns) - set(EXPECTED_COLUMNS)
)

column_order_matches = (
    actual_columns == EXPECTED_COLUMNS
)

print("Missing columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)
print("Column order matches:", column_order_matches)

Missing columns: []
Unexpected columns: []
Column order matches: True


In [16]:
if missing_columns:
    raise ValueError(
        f"Required source columns are missing: {missing_columns}"
    )

In [17]:
hash_columns = EXPECTED_COLUMNS

record_hash_expression = F.sha2(
    F.concat_ws(
        "||",
        *[
            F.coalesce(
                F.col(column).cast("string"),
                F.lit("<NULL>"),
            )
            for column in hash_columns
        ],
    ),
    256,
)

In [18]:
bronze_df = (
    raw_df
    .withColumn(
        "_pipeline_run_id",
        F.lit(pipeline_run_id),
    )
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp(),
    )
    .withColumn(
        "_ingestion_date",
        F.to_date(F.current_timestamp()),
    )
    .withColumn(
        "_source_file",
        F.lit(source_filename),
    )
    .withColumn(
        "_source_file_path",
        F.lit(str(RAW_DATA_PATH)),
    )
    .withColumn(
        "_record_hash",
        record_hash_expression,
    )
)

In [19]:
bronze_df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)
 |-- _pipeline_run_id: string (nullable = false)
 |-- _ingestion_timestamp: timestamp (nullable = false)
 |-- _ingestion_date: date (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- _source_file_path: string (nullable = false)
 |-- _record_hash: string (nullable = true)



In [20]:
print(
    "Raw column count:",
    len(raw_df.columns),
)

print(
    "Bronze column count:",
    len(bronze_df.columns),
)

Raw column count: 11
Bronze column count: 17


In [21]:
bronze_df.select(
    "step",
    "type",
    "amount",
    "nameOrig",
    "nameDest",
    "isFraud",
    "_pipeline_run_id",
    "_ingestion_timestamp",
    "_source_file",
    "_record_hash",
).show(
    n=5,
    truncate=False,
)

+----+--------+--------+-----------+-----------+-------+------------------------------------+--------------------------+------------------------------------+----------------------------------------------------------------+
|step|type    |amount  |nameOrig   |nameDest   |isFraud|_pipeline_run_id                    |_ingestion_timestamp      |_source_file                        |_record_hash                                                    |
+----+--------+--------+-----------+-----------+-------+------------------------------------+--------------------------+------------------------------------+----------------------------------------------------------------+
|1   |PAYMENT |9839.64 |C1231006815|M1979787155|0      |8fb9eff3-1189-40a8-9463-883dbe445a45|2026-07-24 18:22:59.036267|PS_20174392719_1491204439457_log.csv|43363620118f0498b51bb08895f4b298fd12bbb53fbd594d066e2475e837717f|
|1   |PAYMENT |1864.28 |C1666544295|M2044282225|0      |8fb9eff3-1189-40a8-9463-883dbe445a45|2026-07-24 18:2

In [22]:
bronze_record_count = bronze_df.count()

count_reconciliation_passed = (
    source_record_count == bronze_record_count
)

print(
    "Source records:",
    f"{source_record_count:,}",
)

print(
    "Bronze records:",
    f"{bronze_record_count:,}",
)

print(
    "Count reconciliation passed:",
    count_reconciliation_passed,
)

Source records: 6,362,620
Bronze records: 6,362,620
Count reconciliation passed: True


In [23]:
assert count_reconciliation_passed, (
    "Source and Bronze record counts do not match."
)

In [24]:
metadata_columns = [
    "_pipeline_run_id",
    "_ingestion_timestamp",
    "_ingestion_date",
    "_source_file",
    "_source_file_path",
    "_record_hash",
]

In [25]:
metadata_null_expressions = [
    F.sum(
        F.when(
            F.col(column).isNull(),
            1,
        ).otherwise(0)
    ).alias(column)
    for column in metadata_columns
]

metadata_nulls = bronze_df.select(
    *metadata_null_expressions
)

metadata_nulls.show(
    truncate=False
)

+----------------+--------------------+---------------+------------+-----------------+------------+
|_pipeline_run_id|_ingestion_timestamp|_ingestion_date|_source_file|_source_file_path|_record_hash|
+----------------+--------------------+---------------+------------+-----------------+------------+
|0               |0                   |0              |0           |0                |0           |
+----------------+--------------------+---------------+------------+-----------------+------------+



In [26]:
bronze_df.select(
    F.countDistinct(
        "_pipeline_run_id"
    ).alias("distinct_pipeline_run_ids"),
    F.countDistinct(
        "_source_file"
    ).alias("distinct_source_files"),
    F.countDistinct(
        "_ingestion_date"
    ).alias("distinct_ingestion_dates"),
).show()

+-------------------------+---------------------+------------------------+
|distinct_pipeline_run_ids|distinct_source_files|distinct_ingestion_dates|
+-------------------------+---------------------+------------------------+
|                        1|                    1|                       1|
+-------------------------+---------------------+------------------------+



In [27]:
duplicate_hash_df = (
    bronze_df
    .groupBy("_record_hash")
    .agg(
        F.count("*").alias("record_count"),
    )
    .filter(
        F.col("record_count") > 1
    )
)

In [28]:
duplicate_hash_group_count = (
    duplicate_hash_df.count()
)

print(
    "Duplicate hash groups:",
    f"{duplicate_hash_group_count:,}",
)

Duplicate hash groups: 0


In [29]:
duplicate_record_count = (
    duplicate_hash_df
    .select(
        F.sum(
            F.col("record_count") - 1
        ).alias("duplicate_records")
    )
    .first()["duplicate_records"]
)

duplicate_record_count = (
    int(duplicate_record_count)
    if duplicate_record_count is not None
    else 0
)

print(
    "Duplicate records beyond first occurrence:",
    f"{duplicate_record_count:,}",
)

Duplicate records beyond first occurrence: 0


In [30]:
source_null_expressions = [
    F.sum(
        F.when(
            F.col(column).isNull(),
            1,
        ).otherwise(0)
    ).alias(column)
    for column in EXPECTED_COLUMNS
]

source_null_counts = bronze_df.select(
    *source_null_expressions
)

source_null_counts.show(
    truncate=False
)

+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|step|type|amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|0   |0   |0     |0       |0            |0             |0       |0             |0             |0      |0             |
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+



In [31]:
VALID_TRANSACTION_TYPES = [
    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER",
]

In [34]:
bronze_validation_metrics = bronze_df.select(
    F.sum(
        F.when(
            (~F.col("type").isin(VALID_TRANSACTION_TYPES))
            | F.col("type").isNull(),
            1,
        ).otherwise(0)
    ).alias("invalid_transaction_type_count"),

    F.sum(
        F.when(
            (~F.col("step").between(1, 744))
            | F.col("step").isNull(),
            1,
        ).otherwise(0)
    ).alias("invalid_step_count"),

    F.sum(
        F.when(
            (F.col("amount") < 0)
            | F.col("amount").isNull(),
            1,
        ).otherwise(0)
    ).alias("invalid_amount_count"),

    F.sum(
        F.when(
            (~F.col("isFraud").isin(0, 1))
            | F.col("isFraud").isNull(),
            1,
        ).otherwise(0)
    ).alias("invalid_fraud_flag_count"),

    F.sum(
        F.when(
            (~F.col("isFlaggedFraud").isin(0, 1))
            | F.col("isFlaggedFraud").isNull(),
            1,
        ).otherwise(0)
    ).alias("invalid_flagged_fraud_count"),
)

bronze_validation_metrics.show(truncate=False)

+------------------------------+------------------+--------------------+------------------------+---------------------------+
|invalid_transaction_type_count|invalid_step_count|invalid_amount_count|invalid_fraud_flag_count|invalid_flagged_fraud_count|
+------------------------------+------------------+--------------------+------------------------+---------------------------+
|0                             |0                 |0                   |0                       |0                          |
+------------------------------+------------------+--------------------+------------------------+---------------------------+



In [35]:
pipeline_end_timestamp = datetime.now(
    timezone.utc
)

pipeline_duration_seconds = (
    pipeline_end_timestamp
    - pipeline_start_timestamp
).total_seconds()

In [36]:
audit_record = {
    "pipeline_run_id": pipeline_run_id,
    "pipeline_name": "paysim_raw_to_bronze",
    "source_file": source_filename,
    "source_file_path": str(RAW_DATA_PATH),
    "pipeline_start_timestamp": pipeline_start_timestamp.isoformat(),
    "pipeline_end_timestamp": pipeline_end_timestamp.isoformat(),
    "duration_seconds": pipeline_duration_seconds,
    "source_record_count": source_record_count,
    "bronze_record_count": bronze_record_count,
    "duplicate_record_count": duplicate_record_count,
    "count_reconciliation_passed": count_reconciliation_passed,
    "pipeline_status": (
        "SUCCESS"
        if count_reconciliation_passed
        else "FAILED"
    ),
}

In [37]:
audit_df = pd.DataFrame(
    [audit_record]
)

audit_df.T

,0
pipeline_run_id,8fb9eff3-1189-40a8-9463-883dbe445a45
pipeline_name,paysim_raw_to_bronze
source_file,PS_20174392719_1491204439457_log.csv
source_file_path,C:\Projects\paysim-financial-data-pipeline\dat...
pipeline_start_timestamp,2026-07-24T18:20:48.618880+00:00
pipeline_end_timestamp,2026-07-24T18:28:32.371151+00:00
duration_seconds,463.752271
source_record_count,6362620
bronze_record_count,6362620
duplicate_record_count,0


In [38]:
audit_filename = (
    f"raw_to_bronze_audit_"
    f"{pipeline_run_id}.csv"
)

audit_file_path = (
    AUDIT_OUTPUT_PATH
    / audit_filename
)

audit_df.to_csv(
    audit_file_path,
    index=False,
)

print(
    "Audit report written to:",
    audit_file_path,
)

Audit report written to: C:\Projects\paysim-financial-data-pipeline\data\gold\pipeline_audit\raw_to_bronze_audit_8fb9eff3-1189-40a8-9463-883dbe445a45.csv


In [39]:
bronze_df.createOrReplaceTempView(
    "bronze_paysim_transactions"
)

In [40]:
spark.sql(
    """
    SELECT
        type,
        COUNT(*) AS transaction_count,
        SUM(isFraud) AS fraud_count
    FROM bronze_paysim_transactions
    GROUP BY type
    ORDER BY transaction_count DESC
    """
).show()

+--------+-----------------+-----------+
|    type|transaction_count|fraud_count|
+--------+-----------------+-----------+
|CASH_OUT|          2237500|       4116|
| PAYMENT|          2151495|          0|
| CASH_IN|          1399284|          0|
|TRANSFER|           532909|       4097|
|   DEBIT|            41432|          0|
+--------+-----------------+-----------+



## Bronze-Layer Design Decisions

1. All source columns are preserved without renaming or transformation.

2. Operational metadata columns use an underscore prefix to distinguish them
   from source business fields.

3. A pipeline run identifier provides execution-level traceability.

4. The ingestion timestamp records when the platform processed the row, not
   when the simulated transaction occurred.

5. A deterministic SHA-256 record hash is created from all source fields.

6. Duplicate source rows are identified but are not removed in Bronze.

7. Data-quality metrics are calculated, but source records are not corrected
   or quarantined in this layer.

8. The original CSV remains immutable.

9. Physical Bronze Parquet persistence is deferred until Spark runs through
   WSL2, Docker, or another Linux-compatible environment.

## Final Findings

- The raw PaySim source was ingested using an explicit Spark schema.
- The Bronze DataFrame preserved all 11 source columns.
- Six operational metadata columns were added.
- Source and Bronze record counts were reconciled.
- Metadata completeness was validated.
- Deterministic record hashes were created for duplicate detection and
  idempotency.
- Pipeline-level audit information was generated and exported.
- No source records were modified or discarded.

In [41]:
spark.stop()

print("Spark session stopped.")

Spark session stopped.
